# 03 · CNES e o de/para município → região de saúde

Todo o painel agrega por **região de saúde**, e nenhuma base do DATASUS traz esse
agrupamento para o SIH — que só tem código IBGE de município. O CNES, porém, tem a
coluna `REGSAUDE` no cadastro de estabelecimentos.

Este notebook testa se dá para derivar o de/para a partir do CNES, e onde isso falha.

In [ ]:
from pathlib import Path
import pandas as pd, pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
pd.set_option('display.max_columns', 60, 'display.width', 200)

def ler(sistema, arquivo, colunas=None):
    """Le um parquet de data/processed. `colunas` evita carregar as 113/208 colunas inteiras."""
    return pd.read_parquet(PROC / sistema / f'{arquivo}.parquet', columns=colunas)

def ler_varios(sistema, glob='*', colunas=None):
    arqs = sorted((PROC / sistema).glob(f'{glob}.parquet'))
    return pd.concat([pd.read_parquet(a, columns=colunas).assign(_arquivo=a.stem) for a in arqs],
                     ignore_index=True)

sorted(p.name for p in PROC.iterdir())

## Capacidade instalada (KPI 2)

In [ ]:
lt = ler_varios('CNES', 'LTSP*')
for c in ['QT_EXIST','QT_SUS','QT_CONTR']:
    lt[c] = pd.to_numeric(lt[c], errors='coerce')
print(f'{len(lt):,} registros de leito · {lt.CNES.nunique():,} estabelecimentos')
print(f'leitos existentes: {lt.QT_EXIST.sum():,.0f} | SUS: {lt.QT_SUS.sum():,.0f} '
      f'({100*lt.QT_SUS.sum()/lt.QT_EXIST.sum():.1f}%)')
lt.groupby('TP_LEITO')[['QT_EXIST','QT_SUS']].sum().sort_values('QT_SUS', ascending=False)

In [ ]:
st = ler_varios('CNES', 'STSP*', ['CNES','CODUFMUN','REGSAUDE','TP_UNID','TPGESTAO',
                                   'VINC_SUS','NAT_JUR','NIV_HIER'])
print(f'{len(st):,} estabelecimentos · {st.CODUFMUN.nunique()} municípios')
st.head(3)

## Derivando o de/para

Um município deveria pertencer a **uma** região de saúde. Se estabelecimentos do mesmo
município apontarem `REGSAUDE` diferentes, o de/para é ambíguo e precisa da tabela
oficial do DATASUS.

In [ ]:
dp = (st.groupby('CODUFMUN').REGSAUDE
        .agg(regioes='nunique', valores=lambda s: sorted(s.dropna().unique()))
        .sort_values('regioes', ascending=False))
print(f'municípios: {len(dp)} (SP tem 645)')
print(f'regiões de saúde distintas: {st.REGSAUDE.nunique()}')
print(f'municípios com REGSAUDE ambíguo: {(dp.regioes > 1).sum()}')
dp[dp.regioes > 1].head(20)

In [ ]:
# de/para final — moda da REGSAUDE por município
depara = (st.dropna(subset=['REGSAUDE'])
            .groupby('CODUFMUN').REGSAUDE
            .agg(lambda s: s.value_counts().idxmax())
            .rename('regiao_saude').reset_index())
print(f'{len(depara)} municípios mapeados em {depara.regiao_saude.nunique()} regiões')
depara.regiao_saude.value_counts().head(15)

## Cobertura contra o SIH

A pergunta que decide se o de/para serve: **quantas internações ficam sem região?**
Município sem estabelecimento cadastrado não aparece no CNES, mas seus moradores
internam em outro lugar e continuam no SIH por `MUNIC_RES`.

In [ ]:
sih = ler_varios('SIHSUS', 'RDSP26*', ['MUNIC_RES','MUNIC_MOV','N_AIH'])
m = sih.merge(depara, left_on='MUNIC_RES', right_on='CODUFMUN', how='left')
sem = m.regiao_saude.isna()
print(f'internações sem região por MUNIC_RES: {sem.sum():,} ({100*sem.mean():.2f}%)')
print('municípios de residência não mapeados:', m[sem].MUNIC_RES.nunique())
m[sem].MUNIC_RES.value_counts().head(10)

## Veredito

Preencha depois de rodar:

- Municípios de SP cobertos: ___ / 645
- Municípios com `REGSAUDE` ambíguo: ___
- Internações sem região: ___ %

Se a perda passar de ~1% ou houver ambiguidade relevante, **baixe a tabela oficial de
regiões de saúde** (Modalidade *Documentação* na Transferência de Arquivos) em vez de
derivar do CNES. `VW_IPA_REGIAO` depende disso.